In [16]:
import torch
import polars as pl

In [17]:
# loading the dataset from hugging face
from datasets import load_dataset
dataset = load_dataset("mohamed-khalil/ATHAR")

In [18]:
#separating the data into training and testing data
train_data=dataset['train'].to_polars()
test_data=dataset['test'].to_polars()

In [19]:
def tokenize_native(t):
  # /s to match any whitespace
  t=t.with_columns(
    pl.all()
    .str.to_lowercase() # convert the english letters to lower case
    .str.replace_all(r'http\S+|www\S+|@|#', '') # strip out the links and special characters
    .str.replace_all(r'[^\w\s]', ' ') # delete any non alphanumeric word followed by a space
    .str.replace_all(r'\s+', ' ') # delete any consecutive spaces
    .str.strip_chars()
  ).with_columns(
    pl.format('<eos> {} <sos>', pl.col('arabic')), # add beginning and ending of the sentence from right to left
    pl.format('<sos {} <sos>', pl.col('english')) # add beginning and ending of the sentence from left to right
  ).with_columns(
      pl.all().str.split(' ') # tokenize words
  )
  return t

train_data=tokenize_native(train_data)
test_data=tokenize_native(test_data)

In [20]:
# Pad polars list to the maximum one
# find the maximum list
def padding_df(t):
# add padding to the data to ensure they are compatible to be converted to a tensor type
  for col in t.columns:
    max=t.select(pl.col(col)).with_columns(pl.col(col).list.len()).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
        pl.col(col).list.concat(
            pl.lit("<pad>").repeat_by(max-pl.col(col).list.len()) # to ensure compatibleness, repeat by whole size - list len
            # for example if the list len is 3 max is 8 then add only five elements
        ).cast(pl.List(pl.Categorical)).to_physical()) # encode the categorical data to be accepted in torch

  return t

train_data=padding_df(train_data)
test_data=padding_df(test_data)


In [21]:
def cast_Utf(t):
  for col in t.columns:
    max=t.select(pl.col(col)
    ).with_columns(
        pl.col(col).list.len()
        ).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
      pl.col(col).cast(pl.Array(pl.UInt32, shape=(max)))
    )

  return t

train_data=cast_Utf(train_data)
test_data=cast_Utf(test_data)

In [ ]:
##TODO: prepare the training engine
#TODO: prepare the inference engine
# code for inference to get back the data to the original format
train_data.with_columns(pl.all().cast(pl.List(pl.Categorical)))

## Data Analysis and Preprocessing
- in this part you are required to to conduct proper analysis of the above data
- You are also required to preprocess the above data in manner where is ready for modelling

## Modelling Section
- In this part you are required to build two models transformer and Attention based sequence to sequence model.

In [22]:
import torch
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val= train_test_split(train_data['arabic'], train_data['english'], test_size=0.3)
x_train, x_val, y_train, y_val=x_train.to_torch().to(torch.int64),  x_val.to_torch().to(torch.int64), y_train.to_torch().to(torch.int64), y_val.to_torch().to(torch.int64)

## Attention Based Sequence to Sequence Model

In [23]:
import torch.nn as nn

class Encoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
    super(Encoder, self).__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers

    self.dropout=nn.Dropout(p)
    self.embedding=nn.Embedding(num_embeddings=input_size, embedding_dim=embedding_size)
    self.lstm=nn.LSTM(embedding_size, hidden_size, batch_first=True, num_layers=self.num_layers) # maybe you don't need to initialize a dropout at that layer

  def forward(self, x):
    print("getting executed")
      
    output=self.embedding(x)
    print(output)
    output=self.dropout(output)
    output, (hidden, cell)=self.lstm(output)
    return output, hidden, cell

In [24]:
class Decoder(nn.Module):
  def __init__(self, output_size, embedding_size, hidden_size, num_layers, p):
    super(Decoder, self).__init__()
    self.output_size=output_size
    self.embedding_size=embedding_size
    self.num_layers=num_layers
    self.dropout=nn.Dropout(p)
    self.embedding=nn.Embedding(hidden_size, embedding_size)
    self.lstm=nn.LSTM(embedding_size, embedding_size, batch_first=True, num_layers=self.num_layers, bidirectional=True)
    self.out=nn.Linear(embedding_size * 2, output_size)

  def forward(self, input, hidden, cell):
      #unsqueeze the input so the type matches [1, input]
      input=input.unsqueeze(0)
      #inject the data the the embedding layer and then the dropout layer
      embed=self.embedding(input)
      embed=self.dropout(embed) # prevent overfitting
      output, (hidden, cell)=self.lstm(embed, (hidden, cell))
      # use the same stored hidden and cell states to decode the data
      output=self.out(output.squeeze(0))
      return output, hidden, cell

In [25]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq).__init__()
        self.encoder=encoder
        self.decoder=decoder

    def forward(self, enc_input, enc_output, teacher_forcing_ratio):
        # input shape is [batch, sequence_length, input]
        batch_length=enc_input.shape[0]
        seq_length=enc_output.shape[1]
        vocab_size=self.decoder.output_dim
        outputs=torch.zeros(batch_length, seq_length, vocab_size)  
        encoder_output, encoder_hidden, encoder_cell= self.encoder(enc_input)

        decoder_input=enc_output[0, :]
        for i in range (1, seq_length):
            # now we run the decoder
            decoder_output, decoder_hidden, decoder_cell=self.decodeer(decoder_input, encoder_hidden, encoder_cell)
            # each loop returns one word at a time
            # now we add this word to the outputs
            outputs[i]=decoder_output
            top_word=decoder_output.arg_max(1)
            teacher_force=random.random() < teacher_forcing_ratio # if the teacher_force_ration bigger than the generated random then we take the actual output as the next input, else we take the top generated word as the input
            decoder_input=enc_output[i] if teacher_force else top_word
        
        return ouptuts

In [22]:
input_size=x_train.max().item()
embedding_size=64
hidden_size=512
output_size=y_train.shape[0]
num_layers=1
dropout=0.5
encoder=Encoder(input_size, embedding_size, hidden_size, num_layers, dropout)
dncoder=Decoder(output_size, embedding_size, hidden_size, num_layers, dropout)
encoder(x_train)

getting executed


IndexError: index out of range in self